# VPO advantage explorer — using the real `vpo.utils.vpo.vpo_advantage`

<a href="https://colab.research.google.com/github/ryanboldi/vpo/blob/main/notebooks/02_advantage_explorer.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook loads the **actual** `vpo_advantage` function from the repo (the same one veRL calls during
training) and feeds it synthetic `(m, k)` reward matrices so you can build intuition for what the
advantage signal looks like under different settings.

No GPU, no veRL — just the advantage estimator on toy data.


## Setup (Colab)

Clone the repo and add it to the Python path. On a local machine where you've already `pip install -e .`'d
the package, skip this cell.


In [ ]:
# !git clone https://github.com/ryanboldi/vpo.git
# %cd vpo
# !pip install numpy scipy torch  # the only deps vpo.utils.vpo needs
import sys; sys.path.insert(0, '..')   # so we can `import vpo` from this notebook


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from vpo.utils.vpo import vpo_advantage


## Demo 1: How the advantage responds to per-rollout coverage

Two rollouts in one prompt group. Both have m=3 solutions on k=3 objectives. Rollout A is a *generalist*
(all three solutions hit ~0.5 on every objective). Rollout B is a *Pareto specialist set* (each solution
maxes out one objective and is zero on the others — together they cover every corner).

Under uniform Dirichlet `α=1`, VPO rewards B more than A: B's own-pool best-of-m under a random
weight w is `max(w₁, w₂, w₃)` (expected ≈0.61), while A's is always exactly 0.5.


In [ ]:
A = np.array([[0.5, 0.5, 0.5]] * 3)                # generalist
B = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])    # specialist set (covers every corner)

sub_scores = [A.tolist(), B.tolist()]
uid = np.array(['p0', 'p0'])

adv, diag = vpo_advantage(sub_scores, uid, n_selections=512)
print(f'Generalist rollout A:  advantage = {adv[0].item():+.3f}')
print(f'Specialist rollout B:  advantage = {adv[1].item():+.3f}')
print(f'\nPool-wide expected best (across the prompt group): {diag["vpo/pool_expected_best_mean"]:.3f}')


B gets a positive advantage and A negative — exactly what we want. (Note the z-norm: the magnitudes
sum to 0 within the prompt group.)

**Coverage is what's rewarded, not specialization per se.** B only wins because its specialists cover
*every* objective. Add a 4th objective that B ignores (k=4, B still only 3 corners) and the sign flips:
any w concentrated on the uncovered objective sends B's best-of-m toward 0, while the generalist still
collects ≈0.5 everywhere. A specialist set that leaves an objective uncovered loses to a mediocre
generalist — try it by adding a column of zeros to B.


## Demo 2: Concentration parameter α

`VPO_ALPHA` controls Dirichlet concentration. `α<1` sharpens w toward the simplex corners (per-objective
extremists); `α>1` softens it toward the centroid (mean-like).


In [ ]:
import os

deltas = []
alphas = [0.1, 0.3, 1.0, 3.0, 10.0]
for a in alphas:
    os.environ['VPO_ALPHA'] = str(a)
    adv, _ = vpo_advantage(sub_scores, uid, n_selections=2048)
    deltas.append((adv[1] - adv[0]).item())
os.environ.pop('VPO_ALPHA', None)

fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(alphas, deltas, 'o-')
ax.set_xscale('log'); ax.set_xlabel('α (VPO_ALPHA)')
ax.set_ylabel('advantage gap  (specialist − generalist)')
ax.set_title('Specialist preference grows as α → 0')
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


As α drops, the gap widens — with corner-sampled weights the specialist set wins almost every w-round,
so its `E_w[max-of-m]` approaches 1 while the generalist's stays at 0.5. At α=10 the two are nearly
equivalent because all weights look like `(0.25, 0.25, 0.25, 0.25)`.


## Demo 3: Sampler comparison (naive vs Sobol)

Both samplers cover the simplex; Sobol does so quasi-uniformly. For a fixed `n_selections`, Sobol has
**lower variance** in the advantage signal — useful when the prompt group is small.


In [ ]:
os.environ.pop('VPO_ALPHA', None)
rng = np.random.default_rng(7)

# Make 8 rollouts with varied (m,k) signals
group = [rng.uniform(0, 1, size=(3, 4)).tolist() for _ in range(8)]
uid = np.array(['p0'] * 8)

# Replicate the advantage 50× with each sampler to measure variance.
results = {}
for sampler in ['naive', 'sobol']:
    os.environ['VPO_SAMPLER'] = sampler
    samples = np.stack([
        vpo_advantage(group, uid, n_selections=64)[0].numpy()
        for _ in range(50)
    ])
    results[sampler] = samples.std(axis=0).mean()
os.environ.pop('VPO_SAMPLER', None)

print(f"Mean per-rollout advantage std over 50 reruns (n_selections=64):")
print(f"  naive: {results['naive']:.4f}")
print(f"  sobol: {results['sobol']:.4f}")
print(f"  variance reduction:  {results['naive']/results['sobol']:.1f}×")


Sobol typically gives 2-6× lower variance for the same compute. The default sampler in production is
`naive`; switch to `sobol` for prompt groups with small `n` (where MC variance dominates).


## Demo 4: Variable k per prompt (LCB-style)

VPO is `k`-agnostic — it inspects `sub_scores[0].shape[1]` per prompt group, so different prompts can
have different `k`. This is critical for code-gen tasks (LCB) where each problem has a different number
of test cases.


In [ ]:
# Three prompts, each with its own k and m
prompt_a = [np.random.rand(3, 4).tolist() for _ in range(4)]   # k=4
prompt_b = [np.random.rand(3, 7).tolist() for _ in range(4)]   # k=7
prompt_c = [np.random.rand(3, 12).tolist() for _ in range(4)] # k=12

sub_scores = prompt_a + prompt_b + prompt_c
uid = np.array(['pa']*4 + ['pb']*4 + ['pc']*4)
adv, diag = vpo_advantage(sub_scores, uid)
print(f'advantages shape: {adv.shape}')                 # (12,)
print(f'pool_size_mean (avg across 3 prompts): {diag["vpo/pool_size_mean"]:.1f}')


That's the whole API surface: `vpo_advantage(sub_scores, uid)` → `(advantages, diagnostics)`.
The advantage tensor is what veRL multiplies through the response mask and hands to PPO.

→ Next up: [`03_train_maze_colab.ipynb`](03_train_maze_colab.ipynb) actually trains a model end-to-end.
